# Menjalankan Flask Web dengan Ngrok di Google Colab

Notebook ini untuk menjalankan aplikasi Flask klasifikasi penyakit strawberry dengan port forwarding via **ngrok** atau **localtunnel**.

## Cara Pakai:
1. Zip folder project ini, lalu upload ke Colab
2. Upload model `googlenet_model.pth` dan `class_names.json` ke folder `models/` (atau sertakan dalam zip)
3. Jalankan semua cell
4. Buka URL yang muncul untuk mengakses web app

**Catatan:** Kaggle memiliki batasan untuk server yang berjalan lama. Disarankan gunakan **Google Colab**.

## 1. Upload & Extract Project

In [ ]:
# Untuk Colab: Upload zip project
import os
try:
    from google.colab import files
    print('Upload file zip project (klasifikasi_penyakit_tanaman_strawberry.zip)')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    os.system(f'unzip -q -o "{zip_name}" -d /content/')
    print('Extract selesai!')
    PROJECT_DIR = '/content/klasifikasi_penyakit_tanaman_strawberry'
except ImportError:
    # Kaggle: Add project sebagai dataset, atau sesuaikan path
    PROJECT_DIR = '/kaggle/working/klasifikasi_penyakit_tanaman_strawberry'
    print('Mode Kaggle. Pastikan project ada di working directory.')

# Cek struktur
if os.path.exists(PROJECT_DIR):
    os.system(f'ls -la {PROJECT_DIR}/')
else:
    print('Folder project tidak ditemukan. Pastikan zip berisi folder klasifikasi_penyakit_tanaman_strawberry')

## 2. Upload Model (jika belum ada di zip)

In [ ]:
# Upload model dan class_names jika belum ada
from google.colab import files
import os

PROJECT_DIR = '/content/klasifikasi_penyakit_tanaman_strawberry'
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

model_path = os.path.join(MODELS_DIR, 'googlenet_model.pth')
classes_path = os.path.join(MODELS_DIR, 'class_names.json')

if not os.path.exists(model_path):
    print('Upload googlenet_model.pth')
    uploaded = files.upload()
    for fn in uploaded:
        if 'model' in fn.lower() or fn.endswith('.pth'):
            os.rename(fn, model_path)
            print(f'Saved to {model_path}')
else:
    print('Model sudah ada')

if not os.path.exists(classes_path):
    print('Upload class_names.json')
    uploaded = files.upload()
    for fn in uploaded:
        if 'class' in fn.lower() or fn.endswith('.json'):
            os.rename(fn, classes_path)
            print(f'Saved to {classes_path}')
else:
    print('class_names.json sudah ada')

## 3. Install Dependencies

In [ ]:
!pip install flask pyngrok -q

# PyTorch biasanya sudah terinstall di Colab
# !pip install torch torchvision -q

## 4. Setup Ngrok & Jalankan Flask

In [ ]:
import threading
import subprocess
import time
import os

PROJECT_DIR = '/content/klasifikasi_penyakit_tanaman_strawberry'
PORT = 5000

def run_flask():
    os.chdir(PROJECT_DIR)
    subprocess.run(['python', 'app.py'], env={**os.environ})

# Start Flask di background
thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(6)

# Opsi 1: Pyngrok (daftar gratis di ngrok.com untuk token)
try:
    from pyngrok import ngrok
    # ngrok.set_auth_token('YOUR_TOKEN')  # Uncomment dan isi
    public_url = ngrok.connect(PORT)
    print('\n=== NGROK URL (buka di browser) ===')
    print(public_url)
    print('='*50)
except Exception as e:
    print(f'Ngrok: {e}. Mencoba localtunnel...')
    import subprocess
    subprocess.run(['pip', 'install', 'localtunnel', '-q'], capture_output=True)
    subprocess.Popen(['npx', 'localtunnel', '--port', '5000'])
    time.sleep(3)
    print('Cek output di atas untuk URL (https://xxx.loca.lt)')

print('Flask berjalan. Akses URL di atas!')

## Alternatif: Tanpa Ngrok Auth (Colab)

Colab punya fitur "ngrok" built-in. Jika pyngrok bermasalah, gunakan cell berikut:

In [ ]:
# Alternatif: Colab built-in port forwarding
# from google.colab.output import eval_js
# print(eval_js("google.colab.kernel.invokeFunction('notebook', [{}], {})"))

# Atau gunakan localtunnel (tidak perlu signup):
# !pip install localtunnel -q
# !lt --port 5000 &
# # Akan muncul URL seperti https://xxx.loca.lt